In [ ]:
!pip install google pandas numpy scikit-learn tensorflow matplotlib seaborn mlflow dagshub kagglehub rarfile


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.

In [ ]:
from dataclasses import dataclass
import pandas as pd
from pandas import DataFrame
from sklearn.model_selection import train_test_split
from keras import Sequential, Input
from keras.layers import BatchNormalization, Conv1D, Dropout, LSTM, Dense, MaxPooling1D
from keras.callbacks import EarlyStopping
from keras.optimizers import Adam
from typing import Dict, Any
from graphene import String
from keras import Sequential
import dagshub as dg
from dagshub.data_engine import datasources
import mlflow
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
from mlflow.models import infer_signature
import numpy as np
import io
import os


In [ ]:
from dataclasses import dataclass


@dataclass
class Config:
    EARLYSTOP_CALLBACK = {
        "monitor": "val_loss",
        "patience": 5,
        "restore_best_weights": True,
    }

    MODEL_PARAMS = {
        "epochs": 5,
        "batch_size": 64,
        "validation_split": 0.2,
        "verbose": 1,
    }

    SETUP_PARAM = {
        "learning_rate": 0.001,
        "batch_size": 64,
        "epochs": 100,
        "dropout": 0.3,
        "validation_split": 0.2,
    }


In [ ]:
class DataSplitter:
    @staticmethod
    def split(X, y, test_size=0.2, random_state=42):
        return train_test_split(
            X, y, test_size=test_size, random_state=random_state, stratify=y
        )


## Preprocessor

In [ ]:
class DataCleaner:
    @staticmethod
    def drop_duplicates(df: DataFrame):
        return df.drop_duplicates()

    @staticmethod
    def fill_null(df):
        for col in df.select_dtypes(include="number"):
            df[col] = df[col].fillna(df[col].median())
        for col in df.select_dtypes(include=["object", "category"]):
            df[col] = df[col].fillna(df[col].mode()[0])
        return df


class TargetEncoder:
    def __init__(self, encoder):
        self.encoder = encoder

    def fit(self, y):
        self.encoder.fit(y)

    def transform(self, y):
        return self.encoder.transform(y)

    def fit_transform(self, y):
        return self.encoder.fit_transform(y)


class FeatureScaler:
    def __init__(self, scaler):
        self.scaler = scaler

    def fit(self, X):
        self.scaler.fit(X)

    def transform(self, X):
        return self.scaler.transform(X)

    def fit_transform(self, X):
        return self.scaler.fit_transform(X)


class PreprocessingPipeline:
    def __init__(self, encoder, scaler):
        self.cleaner = DataCleaner()
        self.encoder = TargetEncoder(encoder)
        self.scaler = FeatureScaler(scaler)

    def process(self, X_train, X_test, y_train, y_test):
        X_train = self.cleaner.fill_null(X_train)
        X_test = self.cleaner.fill_null(X_test)

        y_train = self.encoder.fit_transform(y_train)
        y_test = self.encoder.transform(y_test)

        X_train = self.scaler.fit_transform(X_train)
        X_test = self.scaler.transform(X_test)

        return (X_train, X_test, y_train, y_test)


## Trainer

In [ ]:
class ModelConfigure:
    def __init__(self) -> None:
        self.model = None

    def preprocess_data(self, scaled_x_train, scaled_x_test):
        x_train_reshaped = scaled_x_train.reshape(
            scaled_x_train.shape[0],
            scaled_x_train.shape[1],
            1,
        )
        x_test_reshaped = scaled_x_test.reshape(
            scaled_x_test.shape[0],
            scaled_x_test.shape[1],
            1,
        )
        return x_train_reshaped, x_test_reshaped

    def setup_model(self, reshaped_x_train, classes, config: Config) -> Sequential:
        self.config = config
        name_classes = len(classes)
        self.model = Sequential(
            [
                Input(shape=(reshaped_x_train.shape[1:])),
                Conv1D(64, 3, activation="relu"),
                BatchNormalization(),
                Conv1D(128, 3, activation="relu"),
                BatchNormalization(),
                MaxPooling1D(pool_size=2),
                Dropout(config.SETUP_PARAM["dropout"]),
                LSTM(32),
                Dense(32, activation="relu"),
                Dropout(config.SETUP_PARAM["dropout"]),
                Dense(name_classes, activation="softmax"),
            ],
        )
        self.model.compile(
            optimizer=Adam(learning_rate=config.SETUP_PARAM["learning_rate"]),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"],
        )
        self.model.summary()
        return self.model


class ModelTrainer:
    def __init__(self, model, config: Config) -> None:
        self.config = config
        self.model = model

    def train(self, X_train_reshaped, target_train):
        early_stop = EarlyStopping(
            monitor=self.config.EARLYSTOP_CALLBACK["monitor"],
            patience=self.config.EARLYSTOP_CALLBACK["patience"],
            restore_best_weights=self.config.EARLYSTOP_CALLBACK["restore_best_weights"],
        )

        self.model.fit(
            X_train_reshaped,
            target_train,
            epochs=self.config.MODEL_PARAMS["epochs"],
            batch_size=self.config.MODEL_PARAMS["batch_size"],
            validation_split=self.config.MODEL_PARAMS["validation_split"],
            callbacks=[early_stop],
            verbose=self.config.MODEL_PARAMS["verbose"],
        )
        return {"model": self.model, "model_params": self.config.MODEL_PARAMS}


## Evaluation

In [ ]:
class ModelEvaluation:
    def __init__(self, model: Sequential, feature_test, target_test) -> None:
        self.model = model
        self.feature_test = feature_test
        self.target_test = target_test
        self.loss = None
        self.accuracy = None

    def build(self) -> Dict[str, Any]:
        pred = np.argmax(trained_model.predict(reshaped_test_features), axis=1)
        accuracy = self.generate_accuracy(self.feature_test, self.target_test)
        classification_report = self.generate_classification_report(
            self.target_test, pred
        )
        compute_matrix = self.generate_confusion_matrix(self.target_test, pred)
        return {
            "accuracy": accuracy,
            "classification_report": classification_report,
            "confusion_matrix": compute_matrix,
        }

    def generate_accuracy(self, feature_test, target_test):
        self.loss, self.accuracy = self.model.evaluate(
            self.feature_test, self.target_test, verbose="auto"
        )
        return {"accuracy": self.accuracy * 100, "loss": self.loss}

    def generate_classification_report(self, y_test, pred):
        return classification_report(y_test, pred)

    def generate_confusion_matrix(self, y_test, pred):
        return confusion_matrix(y_test, pred)


## Services

In [ ]:
class DagsController:
    def __init__(self) -> None:
        pass

    def init(self, repo_owner: str, repo_name: str, mlflow: bool = False):
        return dg.init(repo_owner=repo_owner, repo_name=repo_name, mlflow=mlflow)

    def upload_dags_files(self, repo_path: str, dir_name: str) -> None:
        return dg.upload_files(repo_path, dir_name, commit_message="dataset upload")

    def create_datasource(
        self,
        repo_path,
        datasource_name: str,
        data_path: str,
    ):
        return datasources.create_datasource(repo_path, datasource_name, data_path)


## Runner LSTM

In [ ]:
if __name__ == "__main__":
    config = Config()
    encoder = LabelEncoder()
    scaler = StandardScaler()

    train_ds = pd.read_csv("/content/drive/MyDrive/project_data/train.csv")
    test_ds = pd.read_csv("/content/drive/MyDrive/project_data/test.csv")

    X_train = train_ds.drop("Activity", axis=1)
    y_train = train_ds["Activity"]

    X_test = test_ds.drop("Activity", axis=1)
    y_test = test_ds["Activity"]

    pipeline = PreprocessingPipeline(LabelEncoder(), StandardScaler())

    X_train, X_test, y_train, y_test = pipeline.process(
        X_train, X_test, y_train, y_test
    )

    encoder.fit(y_train)
    classes = encoder.classes_

    model_config = ModelConfigure()
    reshaped_x_train, reshaped_test_features = model_config.preprocess_data(
        X_train, X_test
    )
    model = model_config.setup_model(reshaped_x_train, classes=classes, config=config)

    trainer = ModelTrainer(model, config)
    trained_value = trainer.train(reshaped_x_train, y_train)
    trained_model = trained_value["model"]
    trained_model_params = trained_value["model_params"]

    evaluate_model = ModelEvaluation(trained_model, reshaped_test_features, y_test)
    report = evaluate_model.build()
    print(f"Accuracy: {report['accuracy']['accuracy']}")
    print(f"Loss: {report['accuracy']['loss']}")
    print(f'Classification Report: {report["classification_report"]}')
    print(f'Confusion Matrix: {report["confusion_matrix"]}')


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 560, 64)        │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 560, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 558, 128)       │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 558, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 279, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 279, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 32)             │        20,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           198 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 47,590 (185.90 KB)

 Trainable params: 47,206 (184.40 KB)

 Non-trainable params: 384 (1.50 KB)

Epoch 1/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 62s 576ms/step - accuracy: 0.5967 - loss: 1.0467 - val_accuracy: 0.4133 - val_loss: 1.6329
Epoch 2/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 75s 510ms/step - accuracy: 0.7659 - loss: 0.5720 - val_accuracy: 0.4922 - val_loss: 1.5503
Epoch 3/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 89s 588ms/step - accuracy: 0.8126 - loss: 0.4663 - val_accuracy: 0.5785 - val_loss: 1.2552
Epoch 4/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 73s 486ms/step - accuracy: 0.8374 - loss: 0.4096 - val_accuracy: 0.7322 - val_loss: 0.7454
Epoch 5/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 46s 499ms/step - accuracy: 0.8463 - loss: 0.3845 - val_accuracy: 0.7893 - val_loss: 0.6039
93/93 ━━━━━━━━━━━━━━━━━━━━ 5s 53ms/step
93/93 ━━━━━━━━━━━━━━━━━━━━ 5s 59ms/step - accuracy: 0.7601 - loss: 0.5758
Accuracy: 76.00950002670288
Loss: 0.5757843852043152
Classification Report:               precision    recall  f1-score   support

           0       1.00      0.95      0.97       537
           1       0.85      0.84      0.85       491
       

In [ ]:
dags = DagsController()
dags.init(repo_owner="cisco248", repo_name="Medicare-Plus-Project", mlflow=True)

mlflow.set_experiment("HAR_CNN")
with mlflow.start_run() as run:

    mlflow.log_params(params=dict(Config.MODEL_PARAMS))
    mlflow.log_params(params=dict(Config.EARLYSTOP_CALLBACK))
    # mlflow.log_params(Config.SETUP_PARAM)

    mlflow.log_param("train_samples", X_train.shape[0])
    mlflow.log_param("test_samples", X_test.shape[0])
    mlflow.log_param("features", X_train.shape[1])
    mlflow.log_param("classes", len(classes))

    mlflow.keras.log_model(
        trained_model,
        artifact_path="cnn_lstm",
        save_exported_model=True,
        registered_model_name="CNN-LSTM",
        signature=infer_signature(
            X_train,
            trained_model.predict(reshaped_test_features),
        ),
    )

    acc_val = report["accuracy"]["accuracy"]
    loss_val = report["accuracy"]["loss"]

    mlflow.log_metrics({"accuracy": acc_val, "loss": loss_val})


Initialized MLflow to track repo "cisco248/Medicare-Plus-Project"

Repository cisco248/Medicare-Plus-Project initialized!

93/93 ━━━━━━━━━━━━━━━━━━━━ 8s 86ms/step


2026/06/11 23:58:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Saved artifact at '/tmp/tmpfo5baivc/model/data/model'. The following endpoints are available:

* Endpoint 'serve'
  inputs (POSITIONAL_OR_KEYWORD): TensorSpec(shape=(None, 562), dtype=tf.float64, name=None)
  training (POSITIONAL_OR_KEYWORD): Literal[None]
  mask (POSITIONAL_OR_KEYWORD): Literal[None]
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  134286479797584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134286479800272: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134286479800848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134286479800464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134286479801232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134286479800080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134286479800656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134286479798736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134286479801616: TensorSpec(shape=(), dtype=tf.resour

Registered model 'CNN-LSTM' already exists. Creating a new version of this model...
2026/06/11 23:58:18 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: CNN-LSTM, version 9
Created version '9' of model 'CNN-LSTM'.


🏃 View run bedecked-stork-352 at: https://dagshub.com/cisco248/Medicare-Plus-Project.mlflow/#/experiments/0/runs/12caf986ba1648dba4ae9fb796daac9f
🧪 View experiment at: https://dagshub.com/cisco248/Medicare-Plus-Project.mlflow/#/experiments/0
